In [1]:
import pandas as pd
import optuna

import sys
sys.path.append("../src")

from crop_model_calibration_and_run import *

## Calibration
Default AquaCrop rice model parameter
crop_params = {
    "CCx": 0.95,
    "CGC": 0.007,
    "CDC": 0.005,
    "Zmax": 0.50,
    "WP": 19.0,
    "WPy": 100.0,
    "HI0": 0.43,
    "p_up1": 0.01,
    "p_up2": 0.50,
    "p_up3": 0.55,
    "fshape_w1": 3.0,
    "fshape_w2": 3.0,
    "fshape_w3": 3.0,
}

### MM rice

In [30]:
gid_harvest_area_df = pd.read_csv("../../Results/1a_CAs/HAs_MM.csv")
gid_harvest_area_df = gid_harvest_area_df[['GID', 'rice']]
gid_harvest_area_df.columns = ["GID", "harvest_area_ha"]

observed_yield_df = pd.read_csv(
    "../../Results/3.1_DetrendedYield/DetrendedYield_MM_rice.csv"
)
observed_yield_df = observed_yield_df[["Year", "DetrendQuadratic"]]
observed_yield_df.columns = ["year", "yield_obs_t_ha"]

gid_harvest_area_selected_df = get_gid_ha_by_coverage(
    gid_harvest_area_df,
    threshold=0.9
)

weather_folder_path = (
    Path("../../Data/AquaCropInputs/DailyClimate") / "MM"
)

gid_list = gid_harvest_area_selected_df["GID"].tolist()
valid_gids, weather_dfs = get_weather_dfs(gid_list, weather_folder_path)

gid_harvest_area_valid_df = gid_harvest_area_selected_df[
    gid_harvest_area_selected_df["GID"].isin(valid_gids)
    ]

# Crop parameters to be calibrated
crop_params = {
    "HI0": 0.43,
    "WP": 19.0,
    "CDC": 0.005,
    "Zmax": 0.5,
}

parameter_space = make_asymmetric_parameter_space(crop_params, 0.7, 1.3)

yield_threshold = 0.01*observed_yield_df["yield_obs_t_ha"].mean()
print(f"Threshold yield (1% mean detrended yield): {yield_threshold:.3f} t/ha")

objective = make_objective(
    gid_harvest_area_df=gid_harvest_area_valid_df,
    weather_df_list=weather_dfs,
    observed_yield_df=observed_yield_df,
    parameter_space=parameter_space,
    crop_name="PaddyRiceGDD",
    planting_date="05/01",
    sim_start_time="1991/01/01",
    sim_end_time="2020/12/31",
    soil_type="ClayLoam",
    yield_threshold=yield_threshold,
)


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=50,
)

best_params = study.best_params
print(best_params)

107 out of 287 grids selected for calibration, accounting for 90% of area.
Total grids provided: 107


[I 2026-07-03 13:45:33,110] A new study created in memory with name: no-name-44030e08-f830-4394-886f-9715c45e8692


Valid grids after data coverage filtering: 95
Threshold yield (1% mean detrended yield): 0.027 t/ha


[I 2026-07-03 13:50:27,379] Trial 0 finished with value: 2.5879989407106327 and parameters: {'HI0': 0.39763135066261956, 'WP': 24.138143093073044, 'CDC': 0.005695981825434215, 'Zmax': 0.529597545259111}. Best is trial 0 with value: 2.5879989407106327.
[I 2026-07-03 13:56:21,336] Trial 1 finished with value: 1.273249757084015 and parameters: {'HI0': 0.34125280923414864, 'WP': 15.078337531832709, 'CDC': 0.0036742508365045982, 'Zmax': 0.6098528437324806}. Best is trial 1 with value: 1.273249757084015.
[I 2026-07-03 14:02:18,511] Trial 2 finished with value: 4.397551080197988 and parameters: {'HI0': 0.4560876730297479, 'WP': 21.37202738687492, 'CDC': 0.003561753482887407, 'Zmax': 0.6409729556485984}. Best is trial 1 with value: 1.273249757084015.
[I 2026-07-03 14:06:55,272] Trial 3 finished with value: 1.848418849534715 and parameters: {'HI0': 0.5157702013265089, 'WP': 15.720665861732346, 'CDC': 0.004045474901621301, 'Zmax': 0.40502135295603015}. Best is trial 1 with value: 1.2732497570840

{'HI0': 0.4922834892179487, 'WP': 13.345314401231967, 'CDC': 0.0061176731246107546, 'Zmax': 0.3528006578851538}


### TH rice

In [29]:
gid_harvest_area_df = pd.read_csv("../../Results/1a_CAs/HAs_TH.csv")
gid_harvest_area_df = gid_harvest_area_df[['GID', 'rice']]
gid_harvest_area_df.columns = ["GID", "harvest_area_ha"]

observed_yield_df = pd.read_csv(
    "../../Results/3.1_DetrendedYield/DetrendedYield_TH_rice.csv"
)
observed_yield_df = observed_yield_df[["Year", "DetrendQuadratic"]]
observed_yield_df.columns = ["year", "yield_obs_t_ha"]

gid_harvest_area_selected_df = get_gid_ha_by_coverage(
    gid_harvest_area_df,
    threshold=0.9
)

weather_folder_path = (
    Path("../../Data/AquaCropInputs/DailyClimate") / "TH"
)

gid_list = gid_harvest_area_selected_df["GID"].tolist()
valid_gids, weather_dfs = get_weather_dfs(gid_list, weather_folder_path)

gid_harvest_area_valid_df = gid_harvest_area_selected_df[
    gid_harvest_area_selected_df["GID"].isin(valid_gids)
    ]

# Crop parameters to be calibrated
crop_params = {
    "HI0": 0.43,
    "WP": 19.0,
    "CDC": 0.005,
    "CCx": 0.95,
}

parameter_space = make_asymmetric_parameter_space(crop_params, 0.5, 1.1)

yield_threshold = 0.01*observed_yield_df["yield_obs_t_ha"].mean()
print(f"Threshold yield (1% mean detrended yield): {yield_threshold:.3f} t/ha")

objective = make_objective(
    gid_harvest_area_df=gid_harvest_area_valid_df,
    weather_df_list=weather_dfs,
    observed_yield_df=observed_yield_df,
    parameter_space=parameter_space,
    crop_name="PaddyRiceGDD",
    planting_date="05/01",
    sim_start_time="1991/01/01",
    sim_end_time="2020/12/31",
    soil_type="ClayLoam",
    yield_threshold=yield_threshold,
)


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=50,
)

best_params = study.best_params
print(best_params)

110 out of 231 grids selected for calibration, accounting for 90% of area.
Total grids provided: 110


[I 2026-07-03 09:28:22,149] A new study created in memory with name: no-name-29245982-165d-4c34-b9c5-55b25b8a05d1


Valid grids after data coverage filtering: 106
Threshold yield (1% mean detrended yield): 0.020 t/ha


[I 2026-07-03 09:33:18,627] Trial 0 finished with value: 0.831799854453711 and parameters: {'HI0': 0.3116313506626195, 'WP': 20.338143093073047, 'CDC': 0.006695981825434215, 'CCx': 0.7833091193614738}. Best is trial 0 with value: 0.831799854453711.
[I 2026-07-03 09:38:37,692] Trial 1 finished with value: 0.07014277691353472 and parameters: {'HI0': 0.2552528092341486, 'WP': 11.27833753183271, 'CDC': 0.004674250836504598, 'CCx': 0.9210807150740916}. Best is trial 1 with value: 0.07014277691353472.
[I 2026-07-03 09:43:52,906] Trial 2 finished with value: 7.246107385781493 and parameters: {'HI0': 0.3700876730297479, 'WP': 17.57202738687492, 'CDC': 0.004561753482887407, 'CCx': 0.9745035738634271}. Best is trial 1 with value: 0.07014277691353472.
[I 2026-07-03 09:49:12,983] Trial 3 finished with value: 0.2635226993655403 and parameters: {'HI0': 0.4297702013265088, 'WP': 11.920665861732349, 'CDC': 0.005045474901621301, 'CCx': 0.5694533225745184}. Best is trial 1 with value: 0.0701427769135347

{'HI0': 0.2869818659451036, 'WP': 16.126485507440485, 'CDC': 0.005707862391921852, 'CCx': 0.5921401480551517}


### KH rice

In [28]:
gid_harvest_area_df = pd.read_csv("../../Results/1a_CAs/HAs_KH.csv")
gid_harvest_area_df = gid_harvest_area_df[['GID', 'rice']]
gid_harvest_area_df.columns = ["GID", "harvest_area_ha"]

observed_yield_df = pd.read_csv(
    "../../Results/3.1_DetrendedYield/DetrendedYield_KH_rice.csv"
)
observed_yield_df = observed_yield_df[["Year", "DetrendQuadratic"]]
observed_yield_df.columns = ["year", "yield_obs_t_ha"]

gid_harvest_area_selected_df = get_gid_ha_by_coverage(
    gid_harvest_area_df,
    threshold=0.9
)

weather_folder_path = (
    Path("../../Data/AquaCropInputs/DailyClimate") / "KH"
)

gid_list = gid_harvest_area_selected_df["GID"].tolist()
valid_gids, weather_dfs = get_weather_dfs(gid_list, weather_folder_path)

gid_harvest_area_valid_df = gid_harvest_area_selected_df[
    gid_harvest_area_selected_df["GID"].isin(valid_gids)
    ]

# Crop parameters to be calibrated
crop_params = {
    "HI0": 0.43,
    "WP": 19.0,
    "CDC": 0.005,
    "CCx": 0.95,
}

parameter_space = make_asymmetric_parameter_space(crop_params, 0.4, 1)

yield_threshold = 0.01*observed_yield_df["yield_obs_t_ha"].mean()
print(f"Threshold yield (1% mean detrended yield): {yield_threshold:.3f} t/ha")

objective = make_objective(
    gid_harvest_area_df=gid_harvest_area_valid_df,
    weather_df_list=weather_dfs,
    observed_yield_df=observed_yield_df,
    parameter_space=parameter_space,
    crop_name="PaddyRiceGDD",
    planting_date="05/01",
    sim_start_time="1991/01/01",
    sim_end_time="2020/12/31",
    soil_type="ClayLoam",
    yield_threshold=yield_threshold,
)


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=50,
)

best_params = study.best_params
print(best_params)

34 out of 85 grids selected for calibration, accounting for 90% of area.
Total grids provided: 34


[I 2026-07-03 08:11:42,605] A new study created in memory with name: no-name-b026cfe4-8b65-4578-8437-3fb521bc0984


Valid grids after data coverage filtering: 32
Threshold yield (1% mean detrended yield): 0.013 t/ha


[I 2026-07-03 08:13:10,445] Trial 0 finished with value: 0.5797202680956893 and parameters: {'HI0': 0.26863135066261956, 'WP': 18.438143093073045, 'CDC': 0.0071959818254342155, 'CCx': 0.7212353359923108}. Best is trial 0 with value: 0.5797202680956893.
[I 2026-07-03 08:14:45,157] Trial 1 finished with value: 0.0610661570950202 and parameters: {'HI0': 0.21225280923414863, 'WP': 9.37833753183271, 'CDC': 0.005174250836504598, 'CCx': 0.873720403091713}. Best is trial 1 with value: 0.0610661570950202.
[I 2026-07-03 08:16:20,565] Trial 2 finished with value: 5.185050650584346 and parameters: {'HI0': 0.3270876730297479, 'WP': 15.67202738687492, 'CDC': 0.005061753482887408, 'CCx': 0.9328486157323367}. Best is trial 1 with value: 0.0610661570950202.
[I 2026-07-03 08:17:53,979] Trial 3 finished with value: 0.09546060139622466 and parameters: {'HI0': 0.38677020132650886, 'WP': 10.020665861732349, 'CDC': 0.005545474901621302, 'CCx': 0.4845405706164573}. Best is trial 1 with value: 0.06106615709502

{'HI0': 0.2504944986835608, 'WP': 13.582223320607511, 'CDC': 0.006295835055926347, 'CCx': 0.5460006099128839}


### VN rice

In [8]:
gid_harvest_area_df = pd.read_csv("../../Results/1a_CAs/HAs_VN.csv")
gid_harvest_area_df = gid_harvest_area_df[['GID', 'rice']]
gid_harvest_area_df.columns = ["GID", "harvest_area_ha"]

observed_yield_df = pd.read_csv(
    "../../Results/3.1_DetrendedYield/DetrendedYield_VN_rice.csv"
)
observed_yield_df = observed_yield_df[["Year", "DetrendQuadratic"]]
observed_yield_df.columns = ["year", "yield_obs_t_ha"]

gid_harvest_area_selected_df = get_gid_ha_by_coverage(
    gid_harvest_area_df,
    threshold=0.9
)

weather_folder_path = (
    Path("../../Data/AquaCropInputs/DailyClimate") / "VN"
)

gid_list = gid_harvest_area_selected_df["GID"].tolist()
valid_gids, weather_dfs = get_weather_dfs(gid_list, weather_folder_path)

gid_harvest_area_valid_df = gid_harvest_area_selected_df[
    gid_harvest_area_selected_df["GID"].isin(valid_gids)
    ]

# Crop parameters to be calibrated
crop_params = {
    "WP": 19.0,
    "HI0": 0.43,
    "CDC": 0.005,
    "CCx": 0.95,
}

parameter_space = make_asymmetric_parameter_space(crop_params, 0.7, 1.3)

yield_threshold = 0.01*observed_yield_df["yield_obs_t_ha"].mean()
print(f"Threshold yield (1% mean detrended yield): {yield_threshold:.3f} t/ha")

objective = make_objective(
    gid_harvest_area_df=gid_harvest_area_valid_df,
    weather_df_list=weather_dfs,
    observed_yield_df=observed_yield_df,
    parameter_space=parameter_space,
    crop_name="PaddyRiceGDD",
    planting_date="05/01",
    sim_start_time="1991/01/01",
    sim_end_time="2020/12/31",
    soil_type="ClayLoam",
    yield_threshold=yield_threshold,
)


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=50,
)

best_params = study.best_params
print(best_params)

35 out of 151 grids selected for calibration, accounting for 90% of area.
Total grids provided: 35


[I 2026-07-13 10:34:17,935] A new study created in memory with name: no-name-d6304eb6-21bf-42e4-81bf-a0a31dae46f9


Valid grids after data coverage filtering: 28
Threshold yield (1% mean detrended yield): 0.031 t/ha


[I 2026-07-13 10:35:40,835] Trial 0 finished with value: 4.774218677451335 and parameters: {'WP': 17.56975735485993, 'HI0': 0.5462842910537584, 'CDC': 0.005695981825434215, 'CCx': 0.8595640073640368}. Best is trial 0 with value: 4.774218677451335.
[I 2026-07-13 10:37:07,685] Trial 1 finished with value: 0.4514524434746715 and parameters: {'WP': 15.078612501043775, 'HI0': 0.3412465862467403, 'CDC': 0.0036742508365045982, 'CCx': 0.9465072473768539}. Best is trial 1 with value: 0.4514524434746715.
[I 2026-07-13 10:38:34,983] Trial 2 finished with value: 14.508094493361504 and parameters: {'WP': 20.15271113387258, 'HI0': 0.48368272507137977, 'CDC': 0.003561753482887407, 'CCx': 0.9802207019526481}. Best is trial 1 with value: 0.4514524434746715.
[I 2026-07-13 10:40:04,491] Trial 3 finished with value: 2.8095421260836817 and parameters: {'WP': 22.78984610512481, 'HI0': 0.35578349055499525, 'CDC': 0.004045474901621301, 'CCx': 0.724606465702366}. Best is trial 1 with value: 0.4514524434746715.

{'WP': 17.418588210018783, 'HI0': 0.3121568657831974, 'CDC': 0.004910466564567102, 'CCx': 0.7197509295595008}


### PH rice

In [2]:
gid_harvest_area_df = pd.read_csv("../../Results/1a_CAs/HAs_PH.csv")
gid_harvest_area_df = gid_harvest_area_df[['GID', 'rice']]
gid_harvest_area_df.columns = ["GID", "harvest_area_ha"]

observed_yield_df = pd.read_csv(
    "../../Results/3.1_DetrendedYield/DetrendedYield_PH_rice.csv"
)
observed_yield_df = observed_yield_df[["Year", "DetrendQuadratic"]]
observed_yield_df.columns = ["year", "yield_obs_t_ha"]

gid_harvest_area_selected_df = get_gid_ha_by_coverage(
    gid_harvest_area_df,
    threshold=0.9
)

weather_folder_path = (
    Path("../../Data/AquaCropInputs/DailyClimate") / "PH"
)

gid_list = gid_harvest_area_selected_df["GID"].tolist()
valid_gids, weather_dfs = get_weather_dfs(gid_list, weather_folder_path)

gid_harvest_area_valid_df = gid_harvest_area_selected_df[
    gid_harvest_area_selected_df["GID"].isin(valid_gids)
    ]

# Crop parameters to be calibrated
crop_params = {
    "HI0": 0.43,
    "WP": 19.0,
    "CDC": 0.005,
    "CCx": 0.95,
}

parameter_space = make_asymmetric_parameter_space(crop_params, 0.7, 1.3)

yield_threshold = 0.01*observed_yield_df["yield_obs_t_ha"].mean()
print(f"Threshold yield (1% mean detrended yield): {yield_threshold:.3f} t/ha")

objective = make_objective(
    gid_harvest_area_df=gid_harvest_area_valid_df,
    weather_df_list=weather_dfs,
    observed_yield_df=observed_yield_df,
    parameter_space=parameter_space,
    crop_name="PaddyRiceGDD",
    user_harvest_year=np.arange(1992, 2021),
    planting_date="09/01",
    sim_start_time="1991/01/01",
    sim_end_time="2020/12/31",
    soil_type="ClayLoam",
    yield_threshold=yield_threshold,
)


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=50,
)

best_params = study.best_params
print(best_params)

87 out of 214 grids selected for calibration, accounting for 90% of area.
Total grids provided: 87


[I 2026-07-14 11:09:02,575] A new study created in memory with name: no-name-9a9d2e4a-112f-4816-b312-e3254fce5fe9


Valid grids after data coverage filtering: 46
Threshold yield (1% mean detrended yield): 0.026 t/ha


[I 2026-07-14 11:11:26,363] Trial 0 finished with value: 14.671617400283706 and parameters: {'HI0': 0.39763135066261956, 'WP': 24.138143093073044, 'CDC': 0.005695981825434215, 'CCx': 0.8595640073640368}. Best is trial 0 with value: 14.671617400283706.
[I 2026-07-14 11:13:54,884] Trial 1 finished with value: 3.053859684289391 and parameters: {'HI0': 0.34125280923414864, 'WP': 15.078337531832709, 'CDC': 0.0036742508365045982, 'CCx': 0.9465072473768539}. Best is trial 1 with value: 3.053859684289391.
[I 2026-07-14 11:16:22,491] Trial 2 finished with value: 33.79722119471571 and parameters: {'HI0': 0.4560876730297479, 'WP': 21.37202738687492, 'CDC': 0.003561753482887407, 'CCx': 0.9802207019526481}. Best is trial 1 with value: 3.053859684289391.
[I 2026-07-14 11:18:57,533] Trial 3 finished with value: 9.996158078766564 and parameters: {'HI0': 0.5157702013265089, 'WP': 15.720665861732346, 'CDC': 0.004045474901621301, 'CCx': 0.724606465702366}. Best is trial 1 with value: 3.053859684289391.
[

{'HI0': 0.34998383959959484, 'WP': 14.718846809485495, 'CDC': 0.006118362574467186, 'CCx': 0.6659716564367263}


### ID rice

In [ ]:
gid_harvest_area_df = pd.read_csv("../../Results/1a_CAs/HAs_ID.csv")
gid_harvest_area_df = gid_harvest_area_df[['GID', 'rice']]
gid_harvest_area_df.columns = ["GID", "harvest_area_ha"]

observed_yield_df = pd.read_csv(
    "../../Results/3.1_DetrendedYield/DetrendedYield_ID_rice.csv"
)
observed_yield_df = observed_yield_df[["Year", "DetrendQuadratic"]]
observed_yield_df.columns = ["year", "yield_obs_t_ha"]

gid_harvest_area_selected_df = get_gid_ha_by_coverage(
    gid_harvest_area_df,
    threshold=0.9
)

weather_folder_path = (
    Path("../../Data/AquaCropInputs/DailyClimate") / "ID"
)

gid_list = gid_harvest_area_selected_df["GID"].tolist()
valid_gids, weather_dfs = get_weather_dfs(gid_list, weather_folder_path)

gid_harvest_area_valid_df = gid_harvest_area_selected_df[
    gid_harvest_area_selected_df["GID"].isin(valid_gids)
    ]

# Crop parameters to be calibrated
crop_params = {
    "HI0": 0.43,
    "WP": 19.0,
    "CDC": 0.005,
    "CCx": 0.95,
}

parameter_space = make_asymmetric_parameter_space(crop_params, 0.7, 1.3)

yield_threshold = 0.01*observed_yield_df["yield_obs_t_ha"].mean()
print(f"Threshold yield (1% mean detrended yield): {yield_threshold:.3f} t/ha")

objective = make_objective(
    gid_harvest_area_df=gid_harvest_area_valid_df,
    weather_df_list=weather_dfs,
    observed_yield_df=observed_yield_df,
    parameter_space=parameter_space,
    crop_name="PaddyRiceGDD",
    planting_date="02/01",
    sim_start_time="1991/01/01",
    sim_end_time="2020/12/31",
    soil_type="ClayLoam",
    yield_threshold=yield_threshold,
)


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=50,
)

best_params = study.best_params
print(best_params)

243 out of 852 grids selected for calibration, accounting for 90% of area.
Total grids provided: 243


[I 2026-07-14 14:11:26,185] A new study created in memory with name: no-name-e6aefea8-4611-4b5c-9921-75c4c720756c


Valid grids after data coverage filtering: 154
Threshold yield (1% mean detrended yield): 0.042 t/ha


[I 2026-07-14 14:19:22,654] Trial 0 finished with value: 5.282788614068591 and parameters: {'HI0': 0.39763135066261956, 'WP': 24.138143093073044, 'CDC': 0.005695981825434215, 'CCx': 0.8595640073640368}. Best is trial 0 with value: 5.282788614068591.
